In [ ]:
!pip install -q openai

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Groq connected successfully!")

Groq connected successfully!


In [ ]:
import json
import functools

from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

SYSTEM = f"""
ROLE:
You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time.
If it isn't in FACTS, say "I'll check with the owner."
Remember the customer's name and order during the chat.

FORMAT:
Under 50 words.
Friendly, English or Telugu matching the customer.
"""

def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        result = func(*args, **kwargs)

        print(
            f"[log] {func.__name__}{args} -> {result!r}"
        )

        return result

    return wrapper


@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""

    return MENU[item.lower().strip()] * qty


history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:

        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:

        reply = "I can't reach the server right now. Please try again shortly."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


print("Total for 6 cupcakes:", get_price("cupcake", 6))


while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):

        print("Bot: Thank you for visiting Sweet Crumbs!")
        break

    print("Bot:", chat(text))

[log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: hi, my name is archana.
Bot: Hi Archana! 👋 Sweet Crumbs Home Bakery is delighted to hear from you. How can I help you today? If you’d like to place an order, just let me know which cake(s) you’re interested in.
You: I want 2 chocolate cakes.
Bot: Hi Archana! Great choice. 2 chocolate cakes at ₹650 each. We need 24‑hour notice. When would you like them delivered? Let me know the date and time that works for you.
You: Can I get eggless cakes?
Bot: Absolutely, Archana! All our cakes, including chocolate, come in eggless versions. Let me know if you’d like the eggless option for both cakes.
You: What is my name?
Bot: Your name is Archana. 🌸
You: What did I order?
Bot: You ordered 2 chocolate cakes (eggless). Let me know the delivery date and time, and we’ll get them ready for you!
You: What is the total price?
Bot: Your total is ₹1,300 for the two chocolate cakes. Let me know the delivery date and time, and we’ll have them

In [ ]:
print(get_price("plum cake", 2))
print(get_price("chocolate cake", 2))
print(get_price("cupcake", 6))

[log] get_price('plum cake', 2) -> 1000
1000
[log] get_price('chocolate cake', 2) -> 1300
1300
[log] get_price('cupcake', 6) -> 480
480


DAY 2 – OBSERVATIONS

1. System Prompt:
It provides the role, facts, rules and format
for the chatbot.

2. Memory:
Conversation history helps the chatbot remember
the customer's name and order.

3. Python Dictionary:
The menu stores item names and prices.

4. Decorator:
The decorator logs the function call and result.

5. Error Handling:
Try and except handle API errors without
immediately crashing the chatbot.

6. History Trimming:
Only recent conversation messages are sent
to reduce unnecessary token usage.